# Earth-to-Mars transfer: expanded velocity sweep and figure diagnostics
This notebook generates Table 2, Figures 3–8, model-validation checks, and the Mars-radius crossing analysis. The main 300-day sweep uses 28.125 s steps (1/128 hour), with the initial state included. The 300-day velocity grid includes the original six trials plus 2,500–3,500 m/s in 100 m/s steps. The day-0 state is included in every closest-distance search. Figure 5 separately uses 3,650 days.


In [1]:

import platform
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

G = 6.67430e-11                 # m^3 kg^-1 s^-2
M_SUN = 1.989e30                # kg
AU = 1.496e11                  # m
MU = G * M_SUN                 # m^3 s^-2
DAY = 86400.0                  # s
R_EARTH = AU
R_MARS = 1.5 * AU
THETA_MARS = np.deg2rad(44.0)
INCREMENTS = np.unique(np.r_[np.arange(0.0, 5001.0, 1000.0),
                                np.arange(2500.0, 3501.0, 100.0)])


def acceleration(position):
    """Solar acceleration for one position or a batch of positions."""
    radius = np.linalg.norm(position, axis=-1, keepdims=True)
    return -MU * position / radius**3


def step(position, velocity, dt):
    """Euler-Cromer: use old position for acceleration, then new velocity for position."""
    velocity += acceleration(position) * dt
    position += velocity * dt


def initial_states(increments=INCREMENTS):
    """Earth at angle 0, Mars 44 deg ahead, spacecraft at Earth's position."""
    earth_p = np.array([R_EARTH, 0.0])
    earth_v = np.array([0.0, np.sqrt(MU / R_EARTH)])
    mars_p = R_MARS * np.array([np.cos(THETA_MARS), np.sin(THETA_MARS)])
    mars_v = np.sqrt(MU / R_MARS) * np.array([-np.sin(THETA_MARS), np.cos(THETA_MARS)])
    craft_p = np.repeat(earth_p[None, :], len(increments), axis=0)
    craft_v = np.column_stack((np.zeros(len(increments)), earth_v[1] + increments))
    return earth_p, earth_v, mars_p, mars_v, craft_p, craft_v


def simulate(increments=INCREMENTS, dt=3600.0, days=300.0, store_path=False):
    """Record t=0, then update all three bodies to the same time before distance.

    The duration must be an exact multiple of dt. Positions and velocities are
    reset for each call. All craft trials share the same planetary trajectories.
    """
    increments = np.asarray(increments, dtype=float)
    n_float = days * DAY / dt
    n = int(round(n_float))
    if not np.isclose(n_float, n, rtol=0, atol=1e-9):
        raise ValueError('Choose a timestep that divides the duration exactly.')
    ep, ev, mp, mv, cp, cv = initial_states(increments)
    distance = np.empty((n + 1, len(increments)))
    craft_radius = np.empty_like(distance)
    energy = np.empty_like(distance)
    craft_path = np.empty((n + 1, len(increments), 2)) if store_path else None
    earth_path = np.empty((n + 1, 2)) if store_path else None
    mars_path = np.empty((n + 1, 2)) if store_path else None
    for k in range(n + 1):
        radius = np.linalg.norm(cp, axis=1)
        distance[k] = np.linalg.norm(cp - mp, axis=1)
        craft_radius[k] = radius
        energy[k] = 0.5 * np.sum(cv**2, axis=1) - MU / radius
        if store_path:
            craft_path[k], earth_path[k], mars_path[k] = cp, ep, mp
        if k < n:
            step(ep, ev, dt)
            step(mp, mv, dt)
            step(cp, cv, dt)
    return dict(increments=increments, dt=dt, days=days, distance=distance,
                craft_radius=craft_radius, energy=energy, craft_path=craft_path,
                earth_path=earth_path, mars_path=mars_path)


def minimum_at(result, days):
    """For every trial, return (minimum distance, time in days, radius in AU)."""
    last = int(round(days * DAY / result['dt']))
    indices = np.argmin(result['distance'][:last + 1], axis=0)
    columns = np.arange(len(indices))
    return (result['distance'][indices, columns], indices * result['dt'] / DAY,
            result['craft_radius'][indices, columns] / AU)


def circular_period(radius):
    return 2 * np.pi * np.sqrt(radius**3 / MU) / DAY


def numerical_period(radius, dt=3600.0):
    """Estimate a full revolution by interpolating the positive-x-axis crossing."""
    p = np.array([radius, 0.0]); v = np.array([0.0, np.sqrt(MU / radius)])
    old_p = p.copy()
    for k in range(1, int(np.ceil(1.05 * circular_period(radius) * DAY / dt)) + 1):
        old_p[:] = p
        step(p, v, dt)
        if old_p[1] < 0 <= p[1] and p[0] > 0:
            fraction = -old_p[1] / (p[1] - old_p[1])
            return ((k - 1) + fraction) * dt / DAY
    raise RuntimeError('No positive-x-axis revolution crossing found.')


def hohmann_benchmark(dt=3600.0):
    a = 0.5 * (R_EARTH + R_MARS)
    transfer_days = np.pi * np.sqrt(a**3 / MU) / DAY
    departure_increment = np.sqrt(MU * (2 / R_EARTH - 1 / a)) - np.sqrt(MU / R_EARTH)
    mars_motion = np.sqrt(MU / R_MARS**3) * transfer_days * DAY
    phase_deg = np.rad2deg(np.pi - mars_motion)
    # At the theoretical arrival time, compare numerical transfer radius and
    # angular phase with the analytical half-ellipse destination (180 degrees).
    result = simulate(np.array([departure_increment]), dt=dt, days=300, store_path=True)
    k = int(round(transfer_days * DAY / dt))
    craft = result['craft_path'][k, 0]
    mars = result['mars_path'][k]
    radius_error = np.linalg.norm(craft) - R_MARS
    phase_error_deg = np.rad2deg(np.arctan2(np.sin(np.arctan2(craft[1], craft[0]) -
                          np.arctan2(mars[1], mars[0])),
                          np.cos(np.arctan2(craft[1], craft[0]) -
                          np.arctan2(mars[1], mars[0]))))
    return dict(transfer_days=transfer_days, departure_increment=departure_increment,
                phase_deg=phase_deg, arrival_radius_error_m=radius_error,
                arrival_angle_separation_deg=phase_error_deg,
                arrival_distance_m=np.linalg.norm(craft - mars))


def run_report():
    print('Python', platform.python_version(), '| NumPy', np.__version__,
          '| Matplotlib', matplotlib.__version__)
    main = simulate(dt=28.125, store_path=False)
    d200, t200, r200 = minimum_at(main, 200)
    d300, t300, r300 = minimum_at(main, 300)
    initial = main['distance'][0, 0]
    assert np.all(d300 <= d200 + 1e-3)
    assert np.all(d300 <= initial + 1e-3)
    print(f'Initial distance: {initial:.6e} m')
    print('\nIncrement (m/s) | min 200 d (m) | min 300 d (m) | day at 300 d min | radius at min (AU)')
    for j, dv in enumerate(INCREMENTS):
        print(f'{dv:6.0f} | {d200[j]:.6e} | {d300[j]:.6e} | {t300[j]:8.3f} | {r300[j]:.6f}')
    print('\nCircular-orbit periods (days): analytical | Euler-Cromer 1 h diagnostic | error')
    for name, radius in [('Earth', R_EARTH), ('Mars', R_MARS)]:
        analytical = circular_period(radius); numerical = numerical_period(radius)
        print(f'{name}: {analytical:.6f} | {numerical:.6f} | {numerical-analytical:+.6f}')
    h = hohmann_benchmark()
    print('\nHohmann benchmark with 1.0 and 1.5 AU circular orbits:')
    for name, value in h.items(): print(f'{name}: {value:.6f}')
    print('\nTimestep convergence for 3,000 m/s over 300 days:')
    reference = None
    for dt in (3600.0, 1800.0, 900.0):
        result = simulate(np.array([3000.0]), dt=dt)
        dist, time, radius = minimum_at(result, 300)
        if reference is None: reference = dist[0]
        e = result['energy'][:, 0]
        drift = np.max(np.abs(e-e[0]))
        print(f'{dt:6.0f} s | min {dist[0]:.6e} m | day {time[0]:.4f} | '
              f'difference from 1 h {dist[0]-reference:+.6e} m | max energy drift {drift:.6e} J/kg')
    print('\nTimestep check for finer-sweep candidate, 2,900 m/s:')
    for dt in (3600.0, 1800.0, 900.0, 450.0, 225.0, 112.5, 56.25, 28.125):
        check = simulate(np.array([2900.0]), dt=dt)
        distance, time, radius = minimum_at(check, 300)
        print(f'{dt:6.0f} s | minimum {distance[0]:.6e} m | day {time[0]:.4f} | radius {radius[0]:.6f} AU')
    # Figure 8: observations only. The right panel expands the finer sweep.
    plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 13, 'xtick.labelsize': 11, 'ytick.labelsize': 11})
    fig, (ax, zoom) = plt.subplots(1, 2, figsize=(10.5, 4.8), layout='constrained')
    for panel in (ax, zoom):
        panel.scatter(INCREMENTS, d300 / 1e9, s=44, color='#1b4965', zorder=3)
        panel.scatter([2900], [d300[np.where(INCREMENTS == 2900)[0][0]] / 1e9],
                      s=82, color='#b2462c', zorder=4)
        panel.grid(axis='y', alpha=.24)
        panel.set_xlabel('Tangential increment (m/s)')
    ax.set(xlim=(-120, 5120), ylim=(0, 80), ylabel='Closest distance to Mars (10⁹ m)',
           title='All tested increments')
    ax.set_xticks(np.arange(0, 5001, 1000))
    zoom.set(xlim=(2450, 3550), ylim=(0, 25), title='Finer sweep')
    zoom.set_xticks([2500, 2700, 2900, 3100, 3300, 3500])
    zoom.annotate('2,900 m/s: 1.42 × 10⁸ m', xy=(2900, d300[np.where(INCREMENTS == 2900)[0][0]] / 1e9),
                  xytext=(3000, 9), arrowprops={'arrowstyle': '-', 'color': '#b2462c'},
                  color='#8f3522', fontsize=11)
    fig.savefig('figure_8_fine_sweep.png', dpi=400, facecolor='white')
    plt.show()
    return main




In [ ]:
results = run_report()


## Interpreting the checks
The comparison prints 200- and 300-day minima and the time and heliocentric radius of each 300-day closest approach. The 28.125 s sweep is compared with successively halved steps; rounding the 2,900 m/s result to about 1.4 × 10⁸ m is more appropriate than treating a sampled minimum as an exact physical prediction. The validation checks compare circular periods, a Hohmann transfer, and timesteps of 3,600, 1,800, and 900 s for the 3,000 m/s trial. Figure 8 shows every tested observation as a point without an interpolated curve. The notebook also calculates total specific orbital energy but does not draw Figure 6.


## Additional figure diagnostics
Run this cell to regenerate Figures 3–7. Figure 5 uses a separate 3,650-day planetary test. Figures 3 and 4 check planetary paths; Figure 6 tests total specific orbital energy at 3,000 m/s; Figure 7 displays the 2,900 m/s trial over 300 days. All use Euler–Cromer updates.


In [ ]:
plt.rcParams.update({'font.size':12,'axes.titlesize':13,'axes.labelsize':12,'xtick.labelsize':11,'ytick.labelsize':11,'legend.fontsize':10,'figure.dpi':180,'savefig.dpi':250})

def planets(days,dt=3600):
    ep,ev,mp,mv,_,_=initial_states(np.array([0.]))
    n=round(days*DAY/dt)
    earth=np.empty((n+1,2));mars=np.empty_like(earth)
    for k in range(n+1):
        earth[k]=ep; mars[k]=mp
        if k<n:
            step(ep,ev,dt);step(mp,mv,dt)
    return earth,mars

# Planet-only validation figures, all integrated with the same Euler–Cromer update.
earth,mars=planets(671)
fig,ax=plt.subplots(figsize=(6,6),layout='constrained')
ax.plot(earth[:round(365.21*24)+1,0]/AU,earth[:round(365.21*24)+1,1]/AU,color='#2077a6',lw=2,label='Earth')
ax.scatter(0,0,s=100,c='#e49a19',label='Sun',zorder=5)
ax.set(xlabel='Heliocentric x (AU)',ylabel='Heliocentric y (AU)',title='Earth orbit, one year',xlim=(-1.13,1.13),ylim=(-1.13,1.13));ax.set_aspect('equal');ax.grid(alpha=.2);ax.legend()
fig.savefig('Figure_3_Revised.png');plt.close(fig)
fig,ax=plt.subplots(figsize=(6,6),layout='constrained')
ax.plot(earth[:,0]/AU,earth[:,1]/AU,color='#2077a6',lw=1.8,label='Earth')
ax.plot(mars[:,0]/AU,mars[:,1]/AU,color='#c94a41',lw=1.8,label='Mars')
ax.scatter(0,0,s=100,c='#e49a19',label='Sun',zorder=5)
ax.set(xlabel='Heliocentric x (AU)',ylabel='Heliocentric y (AU)',title='Planetary orbits over 671 days',xlim=(-1.7,1.7),ylim=(-1.7,1.7));ax.set_aspect('equal');ax.grid(alpha=.2);ax.legend()
fig.savefig('Figure_4_Revised.png');plt.close(fig)

earth,mars=planets(3650)
time=np.arange(len(earth))/24
separation=np.linalg.norm(earth-mars,axis=1)/1e11
fig,ax=plt.subplots(figsize=(9,4.6),layout='constrained')
ax.plot(time,separation,lw=1.8,color='#2077a6')
ax.set(xlabel='Elapsed time (days)',ylabel='Earth–Mars distance (10¹¹ m)',title='Planetary separation over 3,650 days',xlim=(0,3650));ax.grid(alpha=.22)
fig.savefig('Figure_5_Revised.png');plt.close(fig)

fig,(ax,bx)=plt.subplots(2,1,figsize=(8,6),sharex=True,layout='constrained')
for dt,lab,col in [(3600,'3,600 s','#245b82'),(1800,'1,800 s','#e17839'),(900,'900 s','#2a9d8f')]:
    v=simulate(np.array([3000.]),dt=dt,days=300)
    e=v['energy'][:,0];t=np.arange(len(e))*dt/DAY
    ax.plot(t,e/1e7,label=lab,color=col,lw=1.7)
    bx.plot(t,(e-e[0])/1e4,color=col,lw=1.7)
ax.set(ylabel='Total specific energy (10⁷ J/kg)',title='Spacecraft energy, 3,000 m/s diagnostic')
bx.set(xlabel='Elapsed time (days)',ylabel='Change from initial (10⁴ J/kg)',xlim=(0,300))
for a in (ax,bx):a.grid(alpha=.22)
ax.legend(title='Timestep')
fig.savefig('Figure_6_Revised.png');plt.close(fig)

res=simulate(np.array([2900.]),dt=28.125,days=300,store_path=True)
e,m,c=res['earth_path']/AU,res['mars_path']/AU,res['craft_path'][:,0,:]/AU
fig,ax=plt.subplots(figsize=(6.5,6),layout='constrained')
ax.plot(e[:,0],e[:,1],label='Earth',lw=1.8,color='#2077a6')
ax.plot(m[:,0],m[:,1],label='Mars',lw=1.8,color='#c94a41')
ax.plot(c[:,0],c[:,1],label='Spacecraft (2,900 m/s)',lw=2,color='#30815d')
ax.scatter(0,0,s=100,c='#e49a19',label='Sun',zorder=5)
k=np.argmin(res['distance'][:,0]);ax.scatter(c[k,0],c[k,1],s=85,facecolors='none',edgecolors='black',linewidths=1.5,label='Closest approach',zorder=6)
ax.set(xlabel='Heliocentric x (AU)',ylabel='Heliocentric y (AU)',title='Trajectories over 300 days',xlim=(-1.85,1.85),ylim=(-1.85,1.85));ax.set_aspect('equal');ax.grid(alpha=.2);ax.legend(loc='lower left')
fig.savefig('Figure_7_Revised.png');plt.close(fig)


## Mars-radius crossing and angular-position evidence
The following cell interpolates the first outward crossing of 1.5 AU, then compares spacecraft and Mars heliocentric longitude at that same time. The 300-day endpoint and 28.125 s timestep match Table 2.


In [ ]:
run=simulate(np.array([2000.,2900.,3000.,4000.]),dt=28.125,store_path=True)
AU=AU;dt=run['dt'];d=run['distance']; cp=run['craft_path'];mp=run['mars_path'];
for j,dv in enumerate(run['increments']):
 r=np.linalg.norm(cp[:,j,:],axis=1)
 ix=np.where((r[:-1]<1.5*AU)&(r[1:]>=1.5*AU))[0]
 if len(ix):
  k=ix[0];f=(1.5*AU-r[k])/(r[k+1]-r[k]);t=(k+f)*dt/DAY
  c=(1-f)*cp[k,j]+f*cp[k+1,j];m=(1-f)*mp[k]+f*mp[k+1]
  a=(np.degrees(np.arctan2(c[1],c[0]))+360)%360; b=(np.degrees(np.arctan2(m[1],m[0]))+360)%360
  sep=(a-b+180)%360-180
  print(f'{dv:4.0f}: crossing day {t:8.3f}, craft {a:8.3f}°, Mars {b:8.3f}°, separation {sep:+8.3f}°, distance {np.linalg.norm(c-m):.4e} m')
 else: print(f'{dv:4.0f}: no outward crossing through 300 days, max r {np.max(r)/AU:.6f} AU')
idx=np.argmin(d[:,1]);c=cp[idx,1];m=mp[idx];a=np.degrees(np.arctan2(c[1],c[0]));b=np.degrees(np.arctan2(m[1],m[0]));print('2900 closest angular difference',(a-b+180)%360-180)
h=hohmann_benchmark(); print('delta v',2900-h['departure_increment'], 'percent',(2900/h['departure_increment']-1)*100,'time',idx*dt/DAY-h['transfer_days'])
